# Tutorial: Ethernet → ShutterHub → KSC101 shutter cube

Use the Windows Kinesis TCPIP transport to address one KSC101 through a KEH3 over
LAN, then explicitly enable and operate its attached shutter. `ShutterHub` is the
Python router; the physical KEH3 is the network hub. Commands target the **cube's
serial and bay endpoint**, not the hub serial or hub-management port.

ShutterHub now has one actual KEH3/KSC101 passive LAN discovery, identity,
connection and release PASS (E128). Discovery refreshes asynchronous TCPIP
identities using CheckForConnectionChanges after building the SDK list. Ethernet
output actuation and timing remain unqualified. Restart this kernel after updating
the driver; a previously loaded module keeps the old discovery implementation.

## 1. Prepare the LAN and Python kernel

- Install the official Thorlabs Kinesis runtime and project dependencies on Windows:
  run `uv sync --locked` from the repository. Use the project's `.venv` as your
  Jupyter Python kernel (install/select `ipykernel` in that environment if needed).
  Restart the kernel after driver edits.
- Close Kinesis, the Dash GUI, and other programs owning this cube. Match its
  serial to the physical shutter. Disconnect an earlier notebook session before
  rerunning setup; retain the same object if release fails.
- Arrange computer → Ethernet/LAN → powered KEH3 → seated KSC101 → shutter.
  Use the hub's actual configured IPv4 address, reachable from the computer.
  This notebook neither assigns IPs nor changes firmware/firewall/network settings.
- For KEH3 Ethernet operation, disconnect **USB IN** as described in the
  [KEH3 manual, §4.3](https://media.thorlabscms.cn/globalassets/items/k/ke/keh/keh3/doc-103012.pdf?v=0129060351).
  Do not move/redock powered cubes. Establish normal physical interlock,
  independent source protection and approved actuation conditions externally.
  There is no software interlock confirmation gate.

Read [Ethernet qualification notes](../ethernet-assessment.md) and
[commissioning guidance](../HARDWARE_VALIDATION.md). Imports/settings are passive;
discovery and reads contact devices; the later operation cells move hardware.

## 2. Select the KEH3 bay endpoint

| Port on the same KEH3 IPv4 address | Destination |
| --- | --- |
| 40303 | Hub itself; **not** a shutter slot |
| 40304 | Left cube bay |
| 40305 | Middle cube bay |
| 40306 | Right cube bay |
| 40307 | USB OUT device |

This mapping is KEH3-specific. Do not apply it to KEH6 without its documented
mapping; the current driver accepts only these KEH3 cube ports. Python slot `0`
is the notebook's first logical channel; `CUBE_PORT` selects the physical bay.
A reachable TCP port or ping alone does not prove a KSC101 identity/SDK connection.

Set `HUB_IP` and `CUBE_SERIAL` below. The commented address/serial are fictional.
Choose an approved positive `HOLD_SECONDS` only when proceeding to actuation.

In [ ]:
from contextlib import contextmanager
from ipaddress import IPv4Address
from math import isfinite
from time import sleep

from thorlabs_shutter_control import ShutterHub, config

HUB_IP = ""  # Actual KEH3 IPv4 address; example only: "192.0.2.10".
CUBE_PORT = 40304  # Left bay; change to match the actual KEH3 connection.
CUBE_SERIAL = ""  # Actual KSC101 serial; example only: "68000001" (or integer).
HOLD_SECONDS = None  # Enter the approved added Open hold, in seconds.
SLOT = 0

In [ ]:
# Re-running setup must not discard a live or unreleased handle.
if "hub" in globals() and globals()["hub"].get_cached_status(SLOT).has_device:
    raise RuntimeError("Disconnect the existing hub session before rerunning setup")
if not HUB_IP or IPv4Address(HUB_IP).is_unspecified:
    raise ValueError("Set HUB_IP to the actual configured KEH3 IPv4 address")
if type(CUBE_PORT) is not int or CUBE_PORT not in (40304, 40305, 40306, 40307):
    raise ValueError("Select a documented KEH3 cube port, not hub port 40303")
if not CUBE_SERIAL:
    raise ValueError("Set CUBE_SERIAL to the actual KSC101 serial, not the hub serial")
ENDPOINT = f"{HUB_IP}:{CUBE_PORT}"
hub = ShutterHub(
    [CUBE_SERIAL],
    slots=1,
    kinesis_dir=config.KINESIS_DIR,
    ethernet_endpoints=[ENDPOINT],
)
verified = False
output_started = False


@contextmanager
def session_step():
    global verified, output_started
    try:
        yield
    except BaseException:
        # A failed identity never authorizes output commands to that target.
        # Passive discovery/read errors release without changing outputs.
        hub.disconnect(SLOT, close_shutter=verified and output_started)
        verified = False
        output_started = False
        raise


print("Configured LAN-only slot:", SLOT, ENDPOINT, CUBE_SERIAL)

## 3. Discover the cube over LAN — no output writes

`discover()` registers the configured cube endpoint with Kinesis and filters for
KSC101 identities on that route. There is no USB fallback. Expect your selected
cube serial in the list. Hub-only discovery, an empty list or a different serial
is a stop condition; do not try actuation to diagnose it.

In [ ]:
with session_step():
    discovered = hub.discover()
    print("KSC101 serials on selected endpoint:", discovered)
    if str(CUBE_SERIAL) not in discovered:
        raise RuntimeError(
            f"Selected cube {CUBE_SERIAL} absent at {ENDPOINT}; check LAN, bay, "
            "USB IN, competing owners and Kinesis TCPIP support"
        )

## 4. Connect, verify identity, and read feedback

Connect verifies the serial and TCPIP endpoint without enabling or changing
outputs. Explicit getters request feedback afterward. Expected connection is
`connected` and identity equals the selected cube. The driver enforces TCPIP and
endpoint metadata; this is not inferred from USB status. Unknown operating
feedback must be diagnosed before enabling. Feedback is controller information,
not independent blade/optical sensing.

For a passive-only session, skip sections 5–6 and run section 7 to release;
no Close or mode command is sent by that passive path.

In [ ]:
with session_step():
    connected_status = hub.connect(SLOT, CUBE_SERIAL)
    verified = True  # Set only after the driver's identity verification succeeds.
    identity = hub.identify_device(SLOT)
    if identity["serial_number"] != str(CUBE_SERIAL):
        raise RuntimeError("Cube identity changed; stop and investigate")
    status = hub.get_operating_status(SLOT)
    if status.connection != "connected":
        raise RuntimeError("No healthy LAN connection")
    print("Identity:", identity)
    print("LAN endpoint:", hub.ethernet_endpoints[SLOT])
    print("Operating feedback:", status)

## 5. Prepare Manual operation — real hardware commands

Continue only under approved physical operating conditions. Set the hold in
section 2 before proceeding; `None` deliberately prevents accidental Run All
actuation. Disarm sends Inactive, then Manual, before EnableDevice, avoiding a
retained Active/Triggered state. Require Closed/Inactive/Manual feedback and key
enabled. Users manage the physical interlock externally.

Successful steps retain the handle. On an error/Ctrl+C, `session_step()` attempts
Close only after verified actuation has begun and then releases communication.
Cleanup errors remain in the traceback chain; retain the object if release fails.

In [ ]:
with session_step():
    if not verified:
        raise RuntimeError("Run the LAN connection/identity cell first")
    if (
        isinstance(HOLD_SECONDS, bool)
        or not isinstance(HOLD_SECONDS, (int, float))
        or not isfinite(HOLD_SECONDS)
        or HOLD_SECONDS <= 0
    ):
        raise ValueError("Set HOLD_SECONDS to an approved finite positive duration")
    output_started = True  # Cleanup is now allowed to request Close.
    hub.disarm_triggers(SLOT)
    ready = hub.get_operating_status(SLOT)
    if not ready.is_closed_manual or not ready.key_enabled:
        raise RuntimeError("Require Closed/Inactive/Manual feedback and enabled key")
    hub.enable_device(SLOT)
    print("Initial Close:", ready.shutter_state)

## 6. Open → hold → Close in one cell

Open/Close send Active/Inactive only and return `None`; they preserve mode and
enable state. Read feedback separately. Expected feedback is `open`, then `closed`.
The `finally` block attempts Close even if Open feedback or sleep fails. Keeping
this cycle together avoids leaving the shutter Open while pausing between cells.

`HOLD_SECONDS` is an added host wait **after** Open feedback, not a maximum exposure
or calibrated pulse width. LAN/SDK latency and mechanical travel add time. Native
calls can block; kernel termination or network failure can prevent cleanup.
Independent source protection remains necessary. No repetition is performed.

In [ ]:
with session_step():
    ready = hub.get_operating_status(SLOT)
    if not verified or not output_started or not ready.is_closed_manual or not ready.key_enabled:
        raise RuntimeError("Run Manual preparation and verify readiness before Open")
    try:
        hub.open_shutter(SLOT)
        opened = hub.get_shutter_state(SLOT)
        print("Open:", opened)
        if opened != "open":
            raise RuntimeError("Open feedback mismatch; stop and investigate")
        sleep(HOLD_SECONDS)
    finally:
        hub.close_shutter(SLOT)
    closed = hub.get_shutter_state(SLOT)
    print("Close:", closed)
    if closed != "closed":
        raise RuntimeError("Close feedback mismatch; use independent protection")

## 7. Release the connection

Run this after a passive check, after the cycle, or to stop a connected session.
After actuation it attempts Close and releases even if Close fails. A passive
session releases without output commands. Release is not proof of closure;
independently verify physical state. If cleanup fails, preserve the traceback
and the same `hub` object; resolve ownership/communication before retrying this cell.

In [ ]:
hub.disconnect(SLOT, close_shutter=verified and output_started)
verified = False
output_started = False
print("LAN connection released. Independently verify physical state.")

## 8. Interpret and troubleshoot

- **No cube discovered:** check the actual IPv4/subnet, KEH3 bay port and USB IN
  arrangement. Compare passive discovery in Kinesis after releasing this notebook.
  If Kinesis also sees only the hub, preserve that observation; do not substitute
  hub port 40303, guess a KEH6 mapping or silently switch to USB.
- **Wrong identity/transport or ownership:** stop; resolve bay/serial selection
  and competing applications. Do not actuate an unverified target.
- **Unknown feedback or Closed/Manual/key prerequisite failure:** stop and diagnose
  explicitly. Disarm is an intentional write; connection alone never repairs state.
- **Read/command/cleanup failure:** use independent source protection, verify physical
  state, preserve exact errors and retained handles. No automatic reconnect/retry
  or firmware change is attempted by this tutorial.

For a bench record, keep the hub/cube model and firmware, Kinesis/Python versions,
physical bay, private address/serial, discovery/identity and feedback results,
approved hold, independently observed motion, and cleanup outcome. Keep private
identifiers out of shared notebook outputs. Saved outputs here are intentionally
empty. Software execution proves routing/control flow only; Ethernet delivery,
physical shutter motion, optical closure and timing need bench observations.

For another shutter, disconnect first, select its serial and documented endpoint,
then rerun from settings. `ShutterHub` can hold multiple explicitly routed slots;
this lesson intentionally operates only one.